---
# Commande pour le rendu : quarto render main.ipynb --execute
# Instructions pour quarto
format:
  html:
    code-fold: true
    embed-resources: true
---

<!-- Header stylé -->
<div style="background: linear-gradient(90deg, #4e54c8, #8f94fb); padding: 30px; border-radius: 12px; color: white; text-align: center; margin-bottom: 20px;">
  <h3 style="margin: 0; font-size: 3em;">🏠🚆Lien entre l'offre de transport et le prix du logement en Île-de-France 🏠🚆</h3>
</div>

# Introduction

Intro à écrire

Le traitement des données IDFM est présenté sur [cette page](idfm.html).

Le traitement des données DVF est présenté sur [cette page](dvf.html).

# Fusion des deux sources de données 

L'approche retenue et d'ajouter au dataframe DVF des métriques indiquant la desserte en transport de chaque point.


## Ouverture des fichiers

In [2]:
# verifie que l'exectution se fait depuis le bon répertoire
import os
if os.getcwd().endswith("notebooks"):
    os.chdir('..')


import geopandas as gpd

# ouverture des fichiers geojson
gdf_idfm_small = gpd.read_file("cache/results/passage_par_arret_synthetique.geojson")
gdf_idfm = gpd.read_file("cache/results/passage_par_arret_full.geojson")
gdf_dvf = gpd.read_file("cache/results/prix_logements.geojson")

# ajout d'une cle principale à gdf_dvf
gdf_dvf['point_id'] = gdf_dvf.index.astype(str)

# gdf_dvf = gdf_dvf.head()



## Calcul des arrets les plus proches

On calcul la distance à l'arret le plus proche pour chaque mode (on ajoute aussi)

In [3]:
# reprojeter en CRS métrique, trouver le point le plus proche et la distance en mètres
gdf_dvf_m = gdf_dvf.to_crs(epsg=3857)
gdf_idfm_small_m = gdf_idfm_small.to_crs(epsg=3857)
gdf_idfm_m = gdf_idfm.to_crs(epsg=3857)

for target in ["bus", "metro", "tramway", "train"]:
    target_df = gdf_idfm_small_m[gdf_idfm_small_m[f"nb_{target}_per_day"] > 0]

    nearest = gpd.sjoin_nearest(
        gdf_dvf_m,
        target_df[['stop_id', 'geometry']],
        how='left',
        distance_col='dist_m'
    )

    # ajouter résultats (distance en m et km, id du stop le plus proche) au GeoDataFrame original
    nearest = nearest.reset_index(drop=True)

    gdf_dvf[f"nearest_{target}_stop_id"] = nearest['stop_id']
    gdf_dvf[f"nearest_{target}_dist_m"] = nearest['dist_m']

# afficher le résultat
gdf_dvf

,adresse,Date mutation,Valeur fonciere,Code_postal,Commune,Surface reelle bati,Surface terrain,Nombre pieces principales,Code commune,Type local,...,geometry,point_id,nearest_bus_stop_id,nearest_bus_dist_m,nearest_metro_stop_id,nearest_metro_dist_m,nearest_tramway_stop_id,nearest_tramway_dist_m,nearest_train_stop_id,nearest_train_dist_m
0,1 ALL ADRIENNE,02/01/2024,153000.00,93250,VILLEMOMBLE,37.0,0.0,2.0,77,Appartement,...,POINT (2.50639 48.89254),0,IDFM:73300,155.389997,IDFM:426280,3222.044283,IDFM:73312,708.300133,IDFM:73297,810.600601
1,1 ALL ANDRE MALRAUX,08/11/2024,234000.00,77370,NANGIS,81.0,232.0,4.0,327,Maison,...,POINT (3.02135 48.55532),1,IDFM:74261,264.083932,IDFM:69884,71707.338011,IDFM:68293,52890.886380,IDFM:62168,1451.228854
2,1 ALL ANDRE MALRAUX,23/02/2024,333000.00,78260,ACHERES,88.0,408.0,4.0,5,Maison,...,POINT (2.06194 48.9531),2,IDFM:65153,251.057114,IDFM:71517,22369.191978,IDFM:480927,6986.766072,IDFM:73604,3325.780821
3,1 ALL ANTOINE GROSSIN,12/07/2024,710250.00,92140,CLAMART,95.0,0.0,4.0,23,Maison,...,POINT (2.27316 48.8121),3,IDFM:70505,261.182603,IDFM:70671,2109.592489,IDFM:70310,2226.866100,IDFM:70505,261.182603
4,1 ALL ARAGON,19/06/2024,159000.00,93290,TREMBLAY-EN-FRANCE,63.0,0.0,4.0,73,Appartement,...,POINT (2.57231 48.95058),4,IDFM:73498,154.031537,IDFM:426280,15289.094394,IDFM:73411,7031.585384,IDFM:73482,1142.816999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72605,99 RUE VIEILLE DU TEMPLE,13/12/2024,444333.34,75003,PARIS,42.0,0.0,2.0,103,Appartement,...,POINT (2.36137 48.86036),72605,IDFM:71246,260.814763,IDFM:71277,633.176408,IDFM:71799,5318.987767,IDFM:474151,1618.908387
72606,99 RUE VOLTAIRE,05/12/2024,644000.00,92800,PUTEAUX,83.0,0.0,4.0,62,Appartement,...,POINT (2.24246 48.88014),72606,IDFM:71383,199.267173,IDFM:71485,1537.090062,IDFM:71422,1121.599798,IDFM:71422,1121.599798
72607,99 RUE VOLTAIRE,23/05/2024,540000.00,92800,PUTEAUX,71.0,0.0,3.0,62,Appartement,...,POINT (2.24246 48.88014),72607,IDFM:71383,199.267173,IDFM:71485,1537.090062,IDFM:71422,1121.599798,IDFM:71422,1121.599798
72608,990 AV PIERRE MENDES FRANCE,17/12/2024,40000.00,77176,SAVIGNY-LE-TEMPLE,27.0,0.0,1.0,445,Appartement,...,POINT (2.57181 48.59446),72608,IDFM:62284,236.050098,IDFM:69884,31771.979832,IDFM:60450,16857.374937,IDFM:62290,1358.563223


## Calcul de l'offre de transport dans un rayon de 1km

In [4]:
# pour chaque point de gdf_dvf, trouver les stops de gdf_idfm_small à <= 100 m
radius_m = 1000  # distance seuil en mètres

# créer des buffers autour des points (CRS métrique déjà présent : gdf_dvf_m, gdf_idfm_small_m)
gdf_dvf_buffers = gdf_dvf_m.copy()
gdf_dvf_buffers['geometry'] = gdf_dvf_m.geometry.buffer(radius_m)

joined = (
# spatial join : stops intersectant les buffers -> on obtient les couples (stop, point)
gpd.sjoin(
    gdf_dvf_buffers[['point_id', 'geometry']],
    gdf_idfm_m[['stop_id', 'route_id', 'nb_stops_per_day', 'route_type', 'geometry']],
    how='left',
    predicate='intersects',
).drop(columns=['index_right', 'geometry'])
.reset_index(drop=True)
)
joined

# calcule le nombre de stops par route en gardant les groupes avec NaN
grouped = joined.groupby(
    ['point_id', 'route_id', 'route_type'],
    dropna=False,
    as_index=False
).agg(
    nb_stops_per_day_route=('nb_stops_per_day', 'max'),
).reset_index(drop=True)

grouped

grouped2 = joined.groupby(['point_id', 'route_type'],
    dropna=False,
    as_index=False).agg(
    passage_journalier=('nb_stops_per_day', 'sum'),
    nb_routes=('route_id', 'nunique'),
    nb_stations=('stop_id', 'nunique'),

).reset_index()

grouped2 

# pivoter grouped2 pour avoir une colonne par type de route
grouped2_pivot = grouped2.pivot_table(
    index='point_id',
    columns='route_type',
    values=['passage_journalier', 'nb_routes', 'nb_stations'],
    dropna=False,
    fill_value=0
)

grouped2_pivot

# renommer les colonnes pour plus de clarté en utilisant la table de conversion
# et arreter multi index 
grouped2_pivot = grouped2_pivot.rename(columns={
    0: "tramway",
    1: "metro",
    2: "train",
    3: "bus",
    6: "IGNORED",
    7: "IGNORED",
})

grouped2_pivot.columns = [
    f"{name}_{mode}_1km"
    for name, mode in grouped2_pivot.columns
]

grouped2_pivot = grouped2_pivot.reset_index()

# drop coloumns that contain '_nan_'
grouped2_pivot = grouped2_pivot.loc[:, ~(grouped2_pivot.columns.str.contains('_nan_') | grouped2_pivot.columns.str.contains('IGNORED'))]


grouped2_pivot


# merger les résultats dans gdf_dvf
gdf_dvf_final = gdf_dvf.merge(
    grouped2_pivot,
    on='point_id',
    how='left'
)
gdf_dvf_final

,adresse,Date mutation,Valeur fonciere,Code_postal,Commune,Surface reelle bati,Surface terrain,Nombre pieces principales,Code commune,Type local,...,nb_routes_train_1km,nb_routes_bus_1km,nb_stations_tramway_1km,nb_stations_metro_1km,nb_stations_train_1km,nb_stations_bus_1km,passage_journalier_tramway_1km,passage_journalier_metro_1km,passage_journalier_train_1km,passage_journalier_bus_1km
0,1 ALL ADRIENNE,02/01/2024,153000.00,93250,VILLEMOMBLE,37.0,0.0,2.0,77,Appartement,...,1.0,7.0,2.0,0.0,1.0,9.0,1142.0,0.0,162.0,1801.0
1,1 ALL ANDRE MALRAUX,08/11/2024,234000.00,77370,NANGIS,81.0,232.0,4.0,327,Maison,...,0.0,14.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,280.0
2,1 ALL ANDRE MALRAUX,23/02/2024,333000.00,78260,ACHERES,88.0,408.0,4.0,5,Maison,...,0.0,8.0,0.0,0.0,0.0,12.0,0.0,0.0,0.0,2305.0
3,1 ALL ANTOINE GROSSIN,12/07/2024,710250.00,92140,CLAMART,95.0,0.0,4.0,23,Maison,...,1.0,12.0,0.0,0.0,1.0,11.0,0.0,0.0,174.0,2911.0
4,1 ALL ARAGON,19/06/2024,159000.00,93290,TREMBLAY-EN-FRANCE,63.0,0.0,4.0,73,Appartement,...,0.0,8.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,1938.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72605,99 RUE VIEILLE DU TEMPLE,13/12/2024,444333.34,75003,PARIS,42.0,0.0,2.0,103,Appartement,...,0.0,17.0,0.0,6.0,0.0,18.0,0.0,5139.0,0.0,6230.0
72606,99 RUE VOLTAIRE,05/12/2024,644000.00,92800,PUTEAUX,83.0,0.0,4.0,62,Appartement,...,0.0,8.0,0.0,0.0,0.0,14.0,0.0,0.0,0.0,3941.0
72607,99 RUE VOLTAIRE,23/05/2024,540000.00,92800,PUTEAUX,71.0,0.0,3.0,62,Appartement,...,0.0,8.0,0.0,0.0,0.0,14.0,0.0,0.0,0.0,3941.0
72608,990 AV PIERRE MENDES FRANCE,17/12/2024,40000.00,77176,SAVIGNY-LE-TEMPLE,27.0,0.0,1.0,445,Appartement,...,0.0,5.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,568.0


## Visualisation sur un exemple

In [5]:
from branca.element import Template, MacroElement
import folium
# affichage des stops within radius on a map for the first point only
df = joined

m = folium.Map(location=[48.84, 2.35], zoom_start=11, tiles="cartodb positron")
first_point = gdf_dvf.iloc[0]
folium.CircleMarker(
    location=[first_point.geometry.y, first_point.geometry.x],
    popup=first_point['point_id'],
    color='blue'
).add_to(m)
for _, row in df[df['point_id'] == first_point['point_id']].iterrows():
    stop = gdf_idfm_small[gdf_idfm_small['stop_id'] == row['stop_id']]

    stop0 = stop.iloc[0]
    route_name = stop0.get('stop_name', None)

    folium.Marker(
        location=[stop0.geometry.y, stop0.geometry.x],
        tooltip=f"Nearby stop:\n{route_name}",
        icon=folium.Icon(color='red', icon='info-sign')
    ).add_to(m)

m

for target in ["bus", "metro", "tramway", "train"]:
    stop_id = first_point.get(f"nearest_{target}_stop_id")
    stop = gdf_idfm_small[gdf_idfm_small['stop_id'] == stop_id]

    stop0 = stop.iloc[0]
    route_name = stop0.get('stop_name', None)

    folium.Marker(
        location=[stop0.geometry.y, stop0.geometry.x],
        tooltip=f"Nearest {target} stop:\n{route_name}",
        icon=folium.Icon(color='green', icon='info-sign')
    ).add_to(m)

# add legend once (do not recreate inside the loop)
legend_html = """
{% macro html(this, kwargs) %}
<div style="position: fixed; 
    bottom: 50px; left: 50px; width: 220px; padding:8px;
    border:2px solid grey; z-index:9999; font-size:14px;
    background-color:white; box-shadow:2px 2px 6px rgba(0,0,0,0.15);
    ">
  <b>Legend</b><br>
  <span style="display:inline-block;width:12px;height:12px;background:blue;border-radius:50%;margin-right:8px;vertical-align:middle;"></span>
    Point (DVF sample)<br>
  <span style="display:inline-block;width:12px;height:12px;background:red;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>
    Nearby stop (within radius)<br>
  <span style="display:inline-block;width:12px;height:12px;background:green;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>
    Nearest stop (by mode)<br>
  <hr style="margin:6px 0"/>
  <small>Hover or click markers for details</small>
</div>
{% endmacro %}
"""
legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)

print(first_point)

m

adresse                                             1 ALL ADRIENNE
Date mutation                                           02/01/2024
Valeur fonciere                                           153000.0
Code_postal                                                  93250
Commune                                                VILLEMOMBLE
Surface reelle bati                                           37.0
Surface terrain                                                0.0
Nombre pieces principales                                      2.0
Code commune                                                    77
Type local                                             Appartement
Valeur foncière au mètre carré                         4135.135135
search                            1 ALL ADRIENNE 93250 VILLEMOMBLE
longitude                                                 2.506387
latitude                                                 48.892543
result_score                                              0.85

In [6]:
# enregistrer le GeoDataFrame final en GeoJSON
gdf_dvf_final.to_file("cache/results/logements_transport_final.geojson", driver="GeoJSON", encoding="utf-8")

NameError: name 'gpd' is not defined